#### Feature Engineering

Dataset:IEEE-CIS Fraud Detection

| Step | Note |
|------|------|
| Memory Optimization | Downcasts data types to reduce memory usage |
| Data Loading | Merges transaction and identity data, sorts by time |
| Baseline Cleaning | Maps email domains, adds time features (hour/day), label encodes categorical variables |
| Domain-Specific Features | Adds aggregated statistics (mean/std transaction amounts per card), spending deviations, transaction velocity, and device-card aggregations |
| Output | Saves baseline and engineered datasets to Parquet files |

In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_DIR = 'data_raw/IEEE-CIS/ieee-fraud-detection'

# 1. MEMORY OPTIMIZATION
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# 2. LOAD DATA
print("Loading data...")
train_trans = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
train_id = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
df = pd.merge(train_trans, train_id, on='TransactionID', how='left')

# Sort by Time (Chronological Splitting Requirement)
df = df.sort_values('TransactionDT').reset_index(drop=True)

# 3. BASELINE CLEANING 
print("Applying baseline cleaning...")
# Email domain mapping
emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum', 'sc.rr.com': 'spectrum', 
          'nycap.rr.com': 'spectrum', 'charter.net': 'spectrum', 'prodigy.net.mx': 'at&t', 
          'protonmail.com': 'protonmail', 'ptd.net': 'penn_telecom', 'aim.com': 'aol', 
          'web.de': 'other', 'icloud.com': 'apple', 'hotmail.com': 'microsoft'}

for c in ['P_emaildomain', 'R_emaildomain']:
    df[c] = df[c].map(emails).fillna('other')

# Time Features
df['Transaction_hour'] = np.floor(df['TransactionDT'] / 3600) % 24
df['Transaction_day'] = np.floor(df['TransactionDT'] / (3600 * 24)) % 7

# Label Encoding
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Create baseline parquet before adding domain features
df_baseline = reduce_mem_usage(df.copy())
df_baseline.to_parquet('data/iceee_baseline.parquet')

# 4. DOMAIN-SPECIFIC FEATURE ENGINEERING (Rubric Requirements)
print("Adding Domain-Specific Features...")

# A. AGGREGATED STATISTICS (User-level behavior)
# Grouping by card1 (often used as a proxy for user)
df['card1_TransactionAmt_mean'] = df.groupby(['card1'])['TransactionAmt'].transform('mean')
df['card1_TransactionAmt_std'] = df.groupby(['card1'])['TransactionAmt'].transform('std')

# B. SPENDING PATTERN DEVIATIONS
# Ratio of current transaction to user average
df['Amt_to_mean_card1'] = df['TransactionAmt'] / df['card1_TransactionAmt_mean']

# C. TRANSACTION VELOCITY
# Count transactions per card1 in a specific timeframe (approximated by index window for simplicity)
df['card1_count'] = df.groupby(['card1'])['TransactionID'].transform('count')

# D. NEW AGGREGATIONS (Cross-referencing Device and Card)
# Number of unique devices used per card
df['card1_Device_nuniq'] = df.groupby(['card1'])['DeviceInfo'].transform('nunique')

# Fill new NaNs and optimize
df = df.fillna(-999)
df_engineered = reduce_mem_usage(df)

# 5. SAVE FINAL DATASETS
print("Saving engineered parquet...")
df_engineered.to_parquet('data/iceee_feature.parquet')

print("Process Complete. Produced: iceee_baseline.parquet and iceee_feature.parquet")

Loading data...
Applying baseline cleaning...
Mem decreased to 928.69 Mb (52.7% reduction)
Adding Domain-Specific Features...
Mem decreased to 937.70 Mb (52.8% reduction)
Saving engineered parquet...
Process Complete. Produced: iceee_baseline.parquet and iceee_feature.parquet


In [1]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_DIR = 'data_raw/IEEE-CIS/ieee-fraud-detection'

# ============================================================
# 1. MEMORY OPTIMIZATION
# ============================================================
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# ============================================================
# 2. LOAD DATA
# ============================================================
print("Loading data...")
train_trans = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
train_id    = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
df = pd.merge(train_trans, train_id, on='TransactionID', how='left')

# Sort chronologically — required for temporal splitting
df = df.sort_values('TransactionDT').reset_index(drop=True)

# ============================================================
# 3. CHRONOLOGICAL TRAIN / VAL SPLIT  ← must happen FIRST
#    Everything below uses only train statistics to avoid
#    any future data leaking into the training features.
# ============================================================
split_idx = int(len(df) * 0.8)
df_train  = df.iloc[:split_idx].copy()
df_val    = df.iloc[split_idx:].copy()
print(f"Split: {len(df_train):,} train rows / {len(df_val):,} val rows")
del df; gc.collect()

# ============================================================
# 4. BASELINE CLEANING  (fit on train, apply to both)
# ============================================================
print("Applying baseline cleaning...")

# -- Email domain mapping --
emails = {
    'gmail.com': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum',
    'sc.rr.com': 'spectrum', 'nycap.rr.com': 'spectrum', 'charter.net': 'spectrum',
    'prodigy.net.mx': 'at&t', 'protonmail.com': 'protonmail', 'ptd.net': 'penn_telecom',
    'aim.com': 'aol', 'web.de': 'other', 'icloud.com': 'apple', 'hotmail.com': 'microsoft'
}
for c in ['P_emaildomain', 'R_emaildomain']:
    df_train[c] = df_train[c].map(emails).fillna('other')
    df_val[c]   = df_val[c].map(emails).fillna('other')

# -- Time features (deterministic, no leakage) --
for df_ in [df_train, df_val]:
    df_['Transaction_hour'] = np.floor(df_['TransactionDT'] / 3600) % 24
    df_['Transaction_day']  = np.floor(df_['TransactionDT'] / (3600 * 24)) % 7

# -- Label Encoding: fit on train, apply to both --
# We collect all object columns, fit an encoder on train (using train+val
# categories to avoid unseen-label errors on val), then transform both.
cat_cols = [col for col in df_train.columns if df_train[col].dtype == 'object']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined unique values so val unseen categories are handled
    combined = pd.concat([df_train[col], df_val[col]], ignore_index=True).astype(str)
    le.fit(combined)
    df_train[col] = le.transform(df_train[col].astype(str))
    df_val[col]   = le.transform(df_val[col].astype(str))
    encoders[col] = le

# Save baseline parquets (before domain-specific features)
print("Saving baseline parquets...")
df_train_base = reduce_mem_usage(df_train.copy())
df_val_base   = reduce_mem_usage(df_val.copy())
df_train_base.to_parquet('data/iceee_baseline_train.parquet')
df_val_base.to_parquet('data/iceee_baseline_val.parquet')
del df_train_base, df_val_base; gc.collect()

# ============================================================
# 5. DOMAIN-SPECIFIC FEATURE ENGINEERING
#    All aggregations computed on TRAIN only, then merged onto
#    val — this is the key fix vs. the original notebook.
# ============================================================
print("Adding Domain-Specific Features...")

# Helper: compute group aggregation on train, merge onto both splits
def add_group_agg(df_tr, df_vl, group_col, value_col, agg_funcs):
    """
    Computes aggregation statistics from df_tr only, then left-merges
    the result onto both df_tr and df_vl. Returns updated dataframes.
    """
    agg = df_tr.groupby(group_col)[value_col].agg(agg_funcs)
    agg.columns = [f'{group_col}_{value_col}_{fn}' for fn in agg_funcs]
    agg = agg.reset_index()
    df_tr = df_tr.merge(agg, on=group_col, how='left')
    df_vl = df_vl.merge(agg, on=group_col, how='left')
    return df_tr, df_vl

# A. AGGREGATED STATISTICS — mean & std transaction amount per card1
df_train, df_val = add_group_agg(df_train, df_val, 'card1', 'TransactionAmt', ['mean', 'std'])

# B. SPENDING PATTERN DEVIATION — ratio of current txn to card's average
#    (card1_TransactionAmt_mean now exists on both splits from step A)
for df_ in [df_train, df_val]:
    df_['Amt_to_mean_card1'] = df_['TransactionAmt'] / df_['card1_TransactionAmt_mean']

# C. TRANSACTION VELOCITY — total txn count per card1 (from train)
count_map = df_train.groupby('card1')['TransactionID'].count().rename('card1_count')
df_train = df_train.merge(count_map.reset_index(), on='card1', how='left')
df_val   = df_val.merge(count_map.reset_index(), on='card1', how='left')

# D. DEVICE DIVERSITY — unique devices per card1 (from train)
device_map = df_train.groupby('card1')['DeviceInfo'].nunique().rename('card1_Device_nuniq')
df_train = df_train.merge(device_map.reset_index(), on='card1', how='left')
df_val   = df_val.merge(device_map.reset_index(), on='card1', how='left')

# Fill NaNs (val rows for unseen card1 values will be NaN → -999)
df_train = df_train.fillna(-999)
df_val   = df_val.fillna(-999)

# ============================================================
# 6. SAVE ENGINEERED DATASETS
# ============================================================
print("Saving engineered parquets...")
df_train_eng = reduce_mem_usage(df_train)
df_val_eng   = reduce_mem_usage(df_val)
df_train_eng.to_parquet('data/iceee_feature_train.parquet')
df_val_eng.to_parquet('data/iceee_feature_val.parquet')

print()
print("Process Complete. Produced:")
print("  data/iceee_baseline_train.parquet")
print("  data/iceee_baseline_val.parquet")
print("  data/iceee_feature_train.parquet")
print("  data/iceee_feature_val.parquet")

Loading data...
Split: 472,432 train rows / 118,108 val rows
Applying baseline cleaning...
Saving baseline parquets...
Mem decreased to 742.95 Mb (52.7% reduction)
Mem decreased to 185.63 Mb (52.8% reduction)
Adding Domain-Specific Features...
Saving engineered parquets...
Mem decreased to 750.16 Mb (52.8% reduction)
Mem decreased to 187.88 Mb (52.7% reduction)

Process Complete. Produced:
  data/iceee_baseline_train.parquet
  data/iceee_baseline_val.parquet
  data/iceee_feature_train.parquet
  data/iceee_feature_val.parquet
